In [ ]:
# Import necessary libraries
import pandas as pd


In [ ]:
# Import datasets
musicinfo_df = pd.read_csv("Music Info.csv")
history_df = pd.read_csv("User Listening History.csv")


In [ ]:
# Information about the two used dataset
print("=== LISTENING HISTORY DATASET ===")
print(f"Aantal interacties: {len(history_df):,}")
print(f"Aantal unieke gebruikers: {history_df['user_id'].nunique():,}")
print(f"Aantal unieke nummers: {history_df['track_id'].nunique():,}")

print("\nKolommen:")
print(history_df.columns.tolist())

print("\n=== MUSIC INFORMATION DATASET ===")
print(f"Aantal nummers: {len(musicinfo_df):,}")
print(f"Aantal unieke artiesten: {musicinfo_df['artist'].nunique():,}")

print("\nKolommen:")
print(musicinfo_df.columns.tolist())

=== LISTENING HISTORY DATASET ===
Aantal interacties: 9,711,301
Aantal unieke gebruikers: 962,037
Aantal unieke nummers: 30,459

Kolommen:
['track_id', 'user_id', 'playcount']

=== MUSIC INFORMATION DATASET ===
Aantal nummers: 50,683
Aantal unieke artiesten: 8,317

Kolommen:
['track_id', 'name', 'artist', 'spotify_preview_url', 'spotify_id', 'tags', 'genre', 'year', 'duration_ms', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature']


In [ ]:
# Collect all unique track IDs from both datasets
history_ids = set(history_df["track_id"])
musicinfo_ids = set(musicinfo_df["track_id"])

# Find the track IDs that appear in both datasets
common_ids = history_ids.intersection(musicinfo_ids)

print("Overlap:", len(common_ids))

Overlap: 30459


In [ ]:
# Filter both datasets to keep only songs with mathcing track IDs
history_df = history_df[history_df["track_id"].isin(common_ids)]
musicinfo_df = musicinfo_df[musicinfo_df["track_id"].isin(common_ids)]

In [ ]:
# Count the number of unique users 
n = len(pd.unique(history_df['user_id']))
print(n)

962037


COLLABORATIVE FILTERING

In [ ]:
# Import cosine similarity for calculating similarity between items
from sklearn.metrics.pairwise import cosine_similarity

# Import csr_matrix for efficient storage of sparse matrices
from scipy.sparse import csr_matrix

In [ ]:
# Convert user IDs and track IDs into categorical variables
user_ids = history_df["user_id"].astype("category")
track_ids = history_df["track_id"].astype("category")

# Assign a numerical code to each track and user
row = track_ids.cat.codes
col = user_ids.cat.codes

# Use play counts as values in the user-track matrix
data = history_df["playcount"]

In [ ]:
# Create a sparse track-user matrix using play counts
matrix = csr_matrix((data, (row, col)))

In [ ]:
# Create a track information dataframe with track IDs as the index
# Keep only the track name and artist columns
track_info = musicinfo_df.set_index("track_id")[["name", "artist"]]

In [ ]:
# Calculate cosine similarity between tracks based on user play counts
collab_sim = cosine_similarity(matrix)

# Get the track IDs in the same order as the matrix rows
track_index = track_ids.cat.categories

# Store the similarity scores in a Dataframe with track IDs as row and column labels
collab_sim_df = pd.DataFrame(
    collab_sim,
    index=track_index,
    columns=track_index
)

In [ ]:
# Import StandardScaler to normalize the feature values
from sklearn.preprocessing import StandardScaler

# Select audio features for each track, using track IDs as the index
content_features = musicinfo_df.set_index("track_id")[
    ["danceability", "energy", "tempo", "valence"]
]

# Ensure the features follow the same track order as the collaborative filtering matrix
content_features = content_features.loc[track_index]

# Standardize the features so they have a mean of 0 and a standard deviation of 1
scaler = StandardScaler()
content_scaled = scaler.fit_transform(content_features)

# Calculate cosine similarity between tracks based on their audio features
content_sim = cosine_similarity(content_scaled)

# Store the similarity scores in a Dataframe with track IDs as row and column labels
content_sim_df = pd.DataFrame(
    content_sim,
    index=track_index,
    columns=track_index
)

In [ ]:
# Define a list of input songs with their titles and artists
songs = [
    ("are you gonna be my girl", "jet"),
    ("the bitch is back", "elton john"),
    ("animal", "neon trees"),
    ("bitter sweet symphony", "the verve"),
    ("mr. brightside", "the killers"),
    ("levels", "avicii"),
    ("this is hardcore", "pulp")
]

# Find the track ID for a song based on its title and artist
def get_track_id(song_name, artist_name):

    #Filter the dataset using case-insensitive matching for the song title and artist
    result = musicinfo_df[
        (musicinfo_df["name"].str.lower() == song_name.lower()) &
        (musicinfo_df["artist"].str.lower() == artist_name.lower())
    ]

    # Return the track ID if a match is found, otherwise return None
    if not result.empty:
        return result.iloc[0]["track_id"]
    else:
        return None

# Create a list to store the track IDs of the input songs    
input_tracks = []

# Look up the track ID for each song in the input list
for song_name, artist_name in songs:

    tid = get_track_id(song_name, artist_name)

    # Add the track ID if the song is found
    if tid:
        input_tracks.append(tid)

    # Print a message if the song cannot be found
    else:
        print(f"Niet gevonden: {song_name} - {artist_name}")

In [ ]:
# Recommend tracks based on a combination of collaborative and content based filtering
def recommend(input_tracks, alpha=0.3, top_n=11):
    # Store the combined recommendation scores for each track
    scores = {}

    # Calculate the similarity scores for each input track
    for track in input_tracks:
        # Skip tracks that are not present in the similarity matrices
        if track not in collab_sim_df.index:
            continue
        if track not in content_sim_df.index:
            continue

        # Get collaborative and content-based similarity scores for the current track
        collab_scores = collab_sim_df.loc[track]
        content_scores = content_sim_df.loc[track]

        # Compare the current track with all other tracks and exclude tracks that are already in the input list
        for other in collab_sim_df.index:
            if other in input_tracks:
                continue

            # Combine both similarity scores using the chosen weights
            score = (
                alpha * collab_scores.get(other, 0) +
                (1 - alpha) * content_scores.get(other, 0)
            )

            # Add the score to the total score for this track
            scores[other] = scores.get(other, 0) + score

    # Return the top N tracks, sorted by score in descending order
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n]

In [ ]:
# Generate track recommendations based on the input tracks
recommendations = recommend(input_tracks)

In [ ]:
# Display the recommended tracks with their artists and similarity scores
for track, score in recommendations:
    # Check whether track information is available
    if track in track_info.index:
        # Retrieve the track name and artist
        name = track_info.loc[track, "name"]
        artist = track_info.loc[track, "artist"]

        # Print the track details and its recommendation score
        print(f"{name} - {artist} ({score:.3f})")

I Only Want to Be With You - Me First and the Gimme Gimmes (2.447)
Winnebago Warrior - Dead Kennedys (2.446)
F**k You - Sleeping with Sirens (2.445)
Break So Easily - The Mighty Mighty Bosstones (2.444)
All I Ask For - Ill Niño (2.438)
Yet Another Day - Armin van Buuren (2.435)
Granite Wall - Bolt Thrower (2.434)
Unbesiegt - Equilibrium (2.431)
Kill the Light - Lacuna Coil (2.430)
Foot Of The Mountain - a-ha (2.429)
Eatin' Ain't Cheatin' - Steel Panther (2.429)
